# Oslo NDVI in R

This notebook fetches an NDVI time series for Oslo from Google Earth Engine and plots it with `ggplot2`.

In [ ]:
library(ggplot2)

gee_python <- "/home/zander.venter/.cache/pypoetry/virtualenvs/pytorch-template-kqCJA8iU-py3.10/bin/python"
script_path <- file.path(getwd(), "scripts", "get_oslo_ndvi.py")

stopifnot(file.exists(gee_python))
stopifnot(file.exists(script_path))

In [ ]:
output_file <- tempfile(fileext = ".csv")
error_file <- tempfile(fileext = ".log")

status <- system2(
  gee_python,
  c(script_path, "--start", "2019-01-01", "--end", "2024-12-31"),
  stdout = output_file,
  stderr = error_file
)

if (!identical(status, 0L)) {
  error_lines <- readLines(error_file, warn = FALSE)
  stop(paste(c("Earth Engine NDVI fetch failed.", error_lines), collapse = "\n"))
}

ndvi_df <- read.csv(output_file, stringsAsFactors = FALSE)
ndvi_df$date <- as.Date(ndvi_df$date)
head(ndvi_df)

In [ ]:
ggplot(ndvi_df, aes(x = date, y = ndvi)) +
  geom_line(color = "forestgreen", linewidth = 0.8) +
  labs(
    title = "Oslo NDVI Time Series",
    subtitle = "MODIS/061/MOD13Q1 mean NDVI around central Oslo",
    x = "Date",
    y = "NDVI"
  ) +
  theme_minimal(base_size = 12)